# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = 'alvaropoveda7@gmail.com'  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
dataset = datasets.load_iris()
print(dataset.feature_names)
print(dataset.data.shape)
print(len(dataset.feature_names))
print(dataset.target.shape)

In [0]:
# TODO: Carga el dataset Iris y explora su estructura.
# Pistas:
# - Usa datasets.load_iris()
# - Crea un DataFrame con dataset.data y dataset.feature_names
# - Añade la columna target/especie
# - Muestra shape, clases, primeras filas y estadísticas descriptivas

df = pd.DataFrame (
    dataset.data, 
    columns = dataset.feature_names

)
df['species'] = dataset.target


In [0]:
print(df.shape) 
print("-----------------------------------------------------")
print(df.head())
print("-----------------------------------------------------")

print(df.columns)
print("-----------------------------------------------------")

print(df.info())
print("-----------------------------------------------------")

print(df.describe())
print("-----------------------------------------------------")

print(df['species'].unique)
print("-----------------------------------------------------")

print(df['species'].value_counts())
print("-----------------------------------------------------")

### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

In [0]:
# TODO: Crea visualizaciones exploratorias del dataset Iris.
# Ideas:
# - Relación entre longitud/ancho del pétalo por especie
# - Distribución de características por especie
# - Boxplots por especie
# - Matriz de correlación

# fig, axes = plt.subplots(2, 2, figsize=(15, 12))
# ...
# plt.show()


# TODO: Escribe tus observaciones clave sobre separabilidad y correlaciones.

sns.scatterplot(
    data= df,
    x= 'petal length (cm)',
    y= 'petal width (cm)',
    hue= 'species'
)

plt.show() #se deduce que petal length y width aumentan proporcionalmenete por lo general
plt.figure(figsize=(8, 6))
sns.histplot(
    data= df,
    x= 'petal length (cm)',
    hue= 'species'
)
plt.figure(figsize=(8, 6)) #Para que no se acoplen las gráficas.
sns.boxplot(
    data=df,
    x='species', #var categórica
    y ='petal length (cm)' #var numérica
)

plt.figure(figsize=(8, 6)) #Para que no se acoplen las gráficas.
sns.histplot(
    data= df,
    x= 'petal width (cm)',
    hue= 'species'
)

In [0]:
df[dataset.feature_names].corr() 

In [0]:
plt.figure(figsize=(8,6))

sns.heatmap(
    df[dataset.feature_names].corr(),
    annot=True
)

plt.show()

## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# TODO: Separa características (X) y etiquetas (y).
X = df[
        [   'sepal length (cm)', 
            'sepal width (cm)',
            'petal length (cm)',
            'petal width (cm)'
        ]
       
    ] #Necesitamos una lista para devolver varias columnas
y = df['species']

X.head()
y.shape

# TODO: Divide el dataset en entrenamiento y prueba.
# Pistas:
# - Usa train_test_split
# - Reserva una parte para test
# - Usa random_state para reproducibilidad
# - Considera stratify=y para mantener la proporción de clases

# X_train, X_test, y_train, y_test = train_test_split(...)

# TODO: Comprueba tamaños y distribución de clases en cada conjunto.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15, stratify=y, random_state = 42)

assert X_train.shape[0] == y_train.shape[0], "No tienen las mismas filas" #Mismo número de muestras
y_train.value_counts()
y_test.value_counts()



## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# TODO: ENTRENAMIENTO CON MLFLOW
# ========================================

# TODO: Activa autologging de MLflow.
mlflow.sklearn.autolog()


# TODO: Inicia un run con un nombre descriptivo.
with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
#     TODO: Define hiperparámetros del árbol.
    max_depth = 4
    max_features = 4
    random_state = 48

#     TODO: Crea y entrena DecisionTreeClassifier.
    dt = DecisionTreeClassifier(
        max_depth = max_depth,
        max_features = max_features,
        random_state = random_state

    )
    dt.fit(X_train, y_train)
#
#     TODO: Genera predicciones para train y test.
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)
#
#     TODO: Calcula métricas de clasificación.
    accuracy_test = accuracy_score (y_test, 
                                    y_pred_test)
    precision_test = precision_score (y_test, 
                                      y_pred_test,
                                      average='macro')
    recall_test = recall_score (y_test, 
                                y_pred_test,
                                average='macro')
    f1_test = f1_score(y_test,
                        y_pred_test,
                        average='macro')

#     TODO: Evalúa con cross-validation.
    cv_scores = cross_val_score(
        dt, 
        X_train,
        y_train,
    )
#
#     TODO: Registra métricas/parámetros relevantes en MLflow si no quedan registrados automáticamente.

#     TODO: Visualiza matriz de confusión y, opcionalmente, el árbol.

#     TODO: Escribe una breve interpretación de los resultados.


## 🎯 Reflexión del Ejercicio

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración del dataset Iris
2. [ ] Separación train/test
3. [ ] Entrenamiento de un Árbol de Decisión
4. [ ] Evaluación con métricas de clasificación
5. [ ] Registro del experimento en MLflow
6. [ ] Interpretación de resultados

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**TODO: Explica con tus palabras:**
- ¿Qué ventajas tiene un árbol de decisión?
- ¿Qué riesgos tiene respecto a overfitting?
- ¿Cómo interpretarías una matriz de confusión multiclase?

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**TODO:** ¿Qué combinación da el mejor balance entre accuracy y overfitting?

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo con `GridSearchCV`.

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara Random Forest, SVM, KNN o Logistic Regression.

### 🎯 Desafío 4: Análisis de Errores

**TODO:** ¿Qué flores se confunden más y por qué?

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa PCA para reducir a 2 dimensiones y visualizar límites de decisión.


In [0]:
# 🎨 DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del ejercicio.

# ========================================
# TODO: DESAFÍO 1 - Optimización manual
# ========================================
configuraciones = [
    {
        "max_depth": 3,
        "max_features": 2,
        "min_samples_split": 10
    },
    {
        "max_depth": 5,
        "max_features": 3,
        "min_samples_split": 4
    },
    {
        "max_depth": 10,
        "max_features": 4,
        "min_samples_split": 2
    }
]
for config in configuraciones:
    # Entrena, evalúa y registra cada configuración en MLflow
    
    dt= DecisionTreeClassifier(
        max_depth = config["max_depth"],
        max_features= config["max_features"],
        min_samples_split= config["min_samples_split"],
        random_state=48
    )
    dt.fit(X_train,y_train)
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)

    accuracy_train = accuracy_score(
        y_train,
        y_pred_train
    )

    accuracy_test = accuracy_score(
        y_test,
        y_pred_test
    )
    diff = accuracy_train - accuracy_test
    
    print("Configuración:", config)
    print("Accuracy train:", accuracy_train)
    print("Accuracy test:", accuracy_test)
    print("Diferencia:", diff)
    print("---------------------------")



In [0]:
# ========================================
# TODO: DESAFÍO 2 - Grid Search
# ========================================
from sklearn.model_selection import GridSearchCV
dt = DecisionTreeClassifier(random_state=48)
param_grid = {
    'max_depth' : [3, 5, 10, None],
    'max_features': [2, 3, 4, 'sqrt'],
    'min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(
    dt,
    param_grid,
    cv=5

)
grid_search.fit(X_train, y_train)

# ========================================
# TODO: DESAFÍO 3 - Comparación de modelos
# ========================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression


modelos = {
    "Decision Tree": DecisionTreeClassifier(random_state=48),
    "Random Forest": RandomForestClassifier(random_state=48),
    "SVM": SVC(random_state=48),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(random_state=48)
}
resultados = []
for nombre, modelo in modelos.items():
    # Entrena y compara métricas
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)

    accuracy = accuracy_score (y_test, y_pred)
    precision = precision_score (y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    # 4. Guardar el resultado
    resultados.append({
    "Modelo": nombre,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

resultados_df = pd.DataFrame(resultados)
resultados_df
# ========================================
# TODO: DESAFÍO 4 - Análisis de errores
# ========================================
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

incorrect_indices = y_test.index[y_test != y_pred]

# TODO: inspecciona las muestras mal clasificadas.
y_pred_series = pd.Series(    #Con ayuda de chat para ver si se confunde el modelo
    y_pred,
    index=y_test.index
)

errores = X_test.loc[incorrect_indices].copy()

errores["Real"] = y_test.loc[incorrect_indices]
errores["Predicha"] = y_pred_series.loc[incorrect_indices]

errores
# ========================================
# TODO: DESAFÍO 5 - PCA y visualización 2D
# ========================================
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

